# 🚀 FaceKey Collector — Kaggle (Zero-Clutter 2-Cell Notebook)
### Dedicated Worker: `kaggle-worker-1` (3 Parallel Workers, 1 Face Only, Turbo 150+ FPS)
- **Cell 1**: Starts multi-worker collection. Every single video, it **automatically packages and saves chunks** to your private cloud dataset (`VijayTheOne/facekey-dataset-chunks`).
- **Cell 2**: **Emergency Save Button**. Run it anytime you want to stop or force-save all data on disk to the cloud immediately.

## ⚡ CELL 1: RUN DATA COLLECTOR (AUTO-SAVES TO CLOUD AFTER EVERY VIDEO)

In [ ]:
# ==============================================================================
# ⚡ CELL 1: START KAGGLE COLLECTOR (CONTINUOUS AUTO-SAVE & ZERO DISK BLOAT)
# ==============================================================================
import os, sys, shutil

# 1. Working directories
DRIVE_DIR = '/kaggle/working/FaceKeyDataset'
for folder in ('chunks', 'checkpoints', 'locks', 'data'):
    os.makedirs(f'{DRIVE_DIR}/{folder}', exist_ok=True)

# 2. Update to latest GitHub repository code (Self-Healing)
%cd /kaggle/working
APP_DIR = '/kaggle/working/FaceKeyAnimation'
if os.path.isdir(os.path.join(APP_DIR, '.git')):
    print('[*] Updating FaceKeyAnimation to latest GitHub main...')
    !cd {APP_DIR} && git fetch --all --prune && git reset --hard origin/main && git clean -fd
else:
    print('[*] Fresh clone of FaceKeyAnimation from GitHub...')
    !rm -rf {APP_DIR}
    !git clone https://github.com/Vijay-1010110/FaceKeyAnimation.git {APP_DIR}

%cd {APP_DIR}

# 3. Fast Cloud Dependencies & Deno JS solver
!pip install -q -U --pre 'yt-dlp[default]' mediapipe opencv-python-headless numpy sounddevice mss av scipy huggingface_hub pysocks
!if ! command -v deno &> /dev/null; then curl -fsSL https://deno.land/install.sh | sh > /dev/null 2>&1 && ln -sf /root/.deno/bin/deno /usr/local/bin/deno; fi

# 4. Dedicated Queue for Kaggle (Zero Overlap with Colab & Lightning)
urls_file = os.path.join(APP_DIR, 'sessions', 'youtube_urls_kaggle.txt')

# 5. Continuous Cloud Auto-Upload Configuration
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN and os.path.exists('/kaggle/working/hf_token.txt'):
    with open('/kaggle/working/hf_token.txt') as f:
        HF_TOKEN = f.read().strip()
if not HF_TOKEN:
    HF_TOKEN = input('Enter Hugging Face Token (or press Enter if saved): ').strip()
    if HF_TOKEN:
        with open('/kaggle/working/hf_token.txt', 'w') as f:
            f.write(HF_TOKEN)

HF_REPO = 'VijayTheOne/facekey-dataset-chunks'

# 6. Launch Collector (Every chunk auto-uploads continuously and clears disk)
print('\n[*] Launching FaceKey Turbo Multi-Worker Kaggle Collector (kaggle-worker-1)...')
!python scripts/cloud_data_collector.py \
    --drive-dir '{DRIVE_DIR}' \
    --worker-id 'kaggle-worker-1' \
    --urls-file '{urls_file}' \
    --chunk-size 1 \
    --quality 480p \
    --num-workers 3 \
    --hf-token '{HF_TOKEN}' \
    --hf-repo '{HF_REPO}'


---
## 🚨 CELL 2: EMERGENCY SAVE & UPLOAD (RUN ANYTIME TO SAVE ALL DATA NOW)
> [!IMPORTANT]
> Run this cell if you stop the collector or before Kaggle session expires. It immediately packs all sessions on disk into chunks and uploads them to your cloud repository.

In [ ]:
# ==============================================================================
# 🚨 CELL 2: 1-CLICK INSTANT RESCUE & SAVE: PACK & UPLOAD ALL KAGGLE DATA NOW
# ==============================================================================
import os, glob
from huggingface_hub import HfApi

HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN and os.path.exists('/kaggle/working/hf_token.txt'):
    with open('/kaggle/working/hf_token.txt') as f:
        HF_TOKEN = f.read().strip()
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = input('Enter Hugging Face Token: ').strip()

HF_REPO = 'VijayTheOne/facekey-dataset-chunks'

print('[1/2] Packaging all unchunked sessions on Kaggle disk...')
!cd /kaggle/working/FaceKeyAnimation && python -c "from scripts.cloud_sync import CloudSyncManager; cs = CloudSyncManager('/kaggle/working/FaceKeyDataset'); cs.pack_sessions_into_chunk('/kaggle/working/FaceKeyAnimation/sessions', 'kaggle-worker-1')"

print('[2/2] Uploading all chunks to private cloud repository...')
api = HfApi(token=HF_TOKEN)
chunks = glob.glob('/kaggle/working/FaceKeyDataset/chunks/*.tar.gz')

if not chunks:
    print('[+] No pending chunks found! All your data is already safe in the cloud.')
else:
    for c in chunks:
        fname = os.path.basename(c)
        fsize = os.path.getsize(c) / (1024 * 1024)
        print(f'[*] Uploading {fname} ({fsize:.1f} MB)...', end=' ', flush=True)
        api.upload_file(path_or_fileobj=c, path_in_repo=f'chunks/{fname}', repo_id=HF_REPO, repo_type='dataset')
        os.remove(c)
        print('[SUCCESS - SAVED!]')
    print('\n[+] ALL KAGGLE DATA IS 100% SAVED TO CLOUD & SAFE FROM TIMEOUT!')
